# Go2 Course Homework: Public Colab Template

This notebook is a lightweight wrapper around the readable course code.

It assumes that the readable homework package is stored in a course GitHub repo.
Unlike the original payload-based notebook, this version keeps the important
source files visible as normal `.py` / `.json` files.

Recommended usage:
1. run setup
2. inspect the environment
3. show the key files
4. restore a checkpoint and render a demo
5. run the public benchmark

## 1. Configure repository URLs

Set `COURSE_REPO_URL` to the GitHub repository that contains this readable
homework package. The repo should contain:
- `train.py`
- `test_policy.py`
- `generate_public_rollout.py`
- `public_eval.py`
- `go2_pg_env/`
- `configs/course_config.json`

In [8]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

import ctypes as _ctypes
import importlib as _importlib
import pathlib as _pathlib

# nvidia-cusparse-cu12 12.5 was compiled against nvJitLink 12.9, but the
# system CUDA 12.4 only ships nvJitLink 12.4. Pre-loading the pip-packaged
# dependency libs with RTLD_GLOBAL ensures the correct symbol versions are
# in the process's global table before the JAX CUDA plugin loads cuSPARSE.
for _pkg, _lib in [
    ("nvjitlink",    "libnvJitLink.so.12"),
    ("cublas",       "libcublasLt.so.12"),
    ("cuda_runtime", "libcudart.so.12"),
]:
    try:
        _m = _importlib.import_module(f"nvidia.{_pkg}")
        _lib_path = str(_pathlib.Path(_m.__path__[0]) / "lib" / _lib)
        _ctypes.CDLL(_lib_path, mode=_ctypes.RTLD_GLOBAL)
    except Exception:
        pass

BASE_DIR = Path("/data/users/zheng/projects/courses/EEA289/go2_deps")
COURSE_REPO_DIR = BASE_DIR / "go2_course_repo"

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git"
COURSE_REPO_BRANCH = "main"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("Running locally.")
print(f"LD_LIBRARY_PATH set: {os.environ.get('LD_LIBRARY_PATH', '')[:80]}...")


Running locally.
LD_LIBRARY_PATH set: /home/zhengm/.conda/envs/robotics/lib/python3.11/site-packages/nvidia/cusolver/l...


## 2. Install system packages and clone repositories

In [9]:
!command -v ffmpeg >/dev/null && echo "ffmpeg ok" || echo "WARNING: ffmpeg not found. Install in terminal: conda install -c conda-forge ffmpeg"
!python -m pip install -q -U pip setuptools wheel
!python -m pip install -q filelock httpx tqdm
!python -m pip install -q "jax[cuda12]==0.10.0"
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q --user -e .
import site
site.addsitedir(site.getusersitepackages())
%cd {COURSE_REPO_DIR}

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
if str(PLAYGROUND_DIR) not in mujoco_playground.__file__:
    raise RuntimeError(f"Expected mujoco_playground to be imported from {PLAYGROUND_DIR}")


Found existing installation: playground 0.2.0
Uninstalling playground-0.2.0:
  Successfully uninstalled playground-0.2.0
+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground fetch --all --tags
Fetching origin
+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/unitree_mujoco fetch --all --tags
Fetching origin


HEAD is now at dd38c28 [JAX] Replace jnp.clip(..., a_min=..., a_max=...) with jnp.clip(..., min=..., max=...).


+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground/mujoco_playground/external_deps/mujoco_menagerie fetch --all --tags
Fetching origin


HEAD is now at 1a37b05 add IMUState support for G1 in readme files


+ git -C /data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground/mujoco_playground/external_deps/mujoco_menagerie checkout 1b86ece576591213e2b666ebf59508454200ca97


HEAD is now at 1b86ece Merge pull request #213 from TetherIA/tetheria/add-aero-hand-open


/data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground
/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo


Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/zhengm/.conda/envs/robotics/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 201, in _version_check
    version = get_version()
              ^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:85: operation cusparseGetProperty(MAJOR_VERSION, &major) failed: The cuSPARSE library was not found.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/zhengm/.conda/envs/robotics/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 487, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/zhengm/.conda/envs/robotics/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 370, in initialize
    _check_cuda_versions(raise_on_first_error = True)
  File "/home/zhengm/.conda/envs/robotics/lib/python3.11/site-packages/jax_plugin

JAX devices: [CpuDevice(id=0)]
JAX backend: cpu
mujoco_playground imported from: /data/users/zheng/projects/courses/EEA289/go2_deps/mujoco_playground/mujoco_playground/__init__.py


## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

In [10]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo


/home/zhengm/.conda/envs/robotics/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Copied 16 assets into /data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo/go2_pg_env/xmls/assets


## 4. Inspect the environment before training

In [11]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_clean",
        3
      ],
      [
        "accelerometer",
        3
      ],
      [
        "gravity_clean",
        3
      ],
      [
        "local_linvel_clean",
        3
      ],
      [
        "global_angvel",
        3
      ],
      [
   

## 5. Read the most important files

In [12]:
%cd {COURSE_REPO_DIR}
!sed -n '1,220p' go2_pg_env/joystick.py

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo
"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import ma

In [13]:
%cd {COURSE_REPO_DIR}
!sed -n '1,220p' train.py

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo
#!/usr/bin/env python3
"""Train the course Go2 locomotion policy with PPO.

Design goals
------------
1. Keep the file small and readable.
2. Preserve the original two-stage training behavior.
3. Make every output artifact explicit:
   - progress.json
   - summary.json
   - resolved_config.json
   - best_checkpoint/manifest.json
"""

from __future__ import annotations

import argparse
import functools
import json
import os
import time
from pathlib import Path
from typing import Any

from course_common import (
    DEFAULT_CONFIG_PATH,
    apply_stage_config,
    build_env_overrides,
    detect_gpu_name,
    ensure_environment_available,
    export_selected_checkpoint,
    get_ppo_config,
    lazy_import_stack,
    load_json,
    resolve_latest_checkpoint_dir,
    save_json,
    set_runtime_env,
    stage_sequence,
    to_jsonable,
)


ROOT = Path(__file__).resolve().parent


def parse_args() -> argparse.Namespace:
    p

In [14]:
%cd {COURSE_REPO_DIR}
!sed -n '1,220p' benchmark_specs.py

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo
#!/usr/bin/env python3
"""Deterministic command scripts used for demo videos and public benchmark runs."""

from __future__ import annotations

from typing import Any

import numpy as np


PUBLIC_EPISODE_LABELS = (
    "forward_only",
    "lateral_only",
    "yaw_only",
    "combined",
)


def build_demo_segments(config: dict[str, Any]) -> list[list[float]]:
    """Return an easy-to-read command script for demo videos.

    The demo is intentionally human-readable:
    stand -> slow forward -> medium forward -> fast forward -> slow forward -> stand.
    """
    demo_cfg = config["demo_rollout"]
    if "segments" in demo_cfg and demo_cfg["segments"]:
        return [[float(x) for x in segment] for segment in demo_cfg["segments"]]
    return [
        [0.0, 0.0, 0.0],
        [0.20, 0.0, 0.0],
        [0.40, 0.0, 0.0],
        [0.60, 0.0, 0.0],
        [0.30, 0.0, 0.0],
        [0.0, 0.0, 0.0],
    ]


def public_command_

## 6. Define a Colab-friendly runtime config

In [15]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)

wrote /data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo/configs/colab_runtime_config.json


## 7. Dry-run the training config

In [20]:
%cd {COURSE_REPO_DIR}
!python train.py --config configs/colab_runtime_config.json --dry-run

/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_h

## 8. Run training

If you are demonstrating the pipeline interactively, it is usually better to
**skip this cell** and instead use a prepared checkpoint. Full training
includes JIT compile time.

In [ ]:
!python train.py #   --config configs/colab_runtime_config.json #   --stage both #   --output-dir artifacts/run_baseline

[run] output_dir=/data/users/zheng/projects/courses/EEA289/go2_deps/go2_course_repo/artifacts/run_default
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5


## 9. Restore a checkpoint and render a deterministic demo

Replace the checkpoint path below with a real trained checkpoint if needed.

In [ ]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_baseline" / "best_checkpoint"
DEMO_DIR = COURSE_REPO_DIR / "artifacts" / "demo_bundle"

!python test_policy.py #   --config configs/colab_runtime_config.json #   --checkpoint-dir {CHECKPOINT_DIR} #   --stage-name stage_2 #   --output-dir {DEMO_DIR}

## 10. Generate the public benchmark rollout

In [ ]:
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle"

!python generate_public_rollout.py #   --config configs/colab_runtime_config.json #   --checkpoint-dir {CHECKPOINT_DIR} #   --stage-name stage_2 #   --output-dir {PUBLIC_DIR} #   --num-episodes 4 #   --render-first-episode

!python public_eval.py #   --config configs/colab_runtime_config.json #   --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} #   --output-json {PUBLIC_DIR / "public_eval.json"}

## 11. Suggested walkthrough order

Recommended order:
1. `inspect_env.py`
2. `go2_pg_env/joystick.py`
3. `configs/course_config.json`
4. `test_policy.py`
5. `generate_public_rollout.py`
6. `public_eval.json`

This order mirrors the main workflow:
**task -> train -> restore -> benchmark**